# ML Lab · Module 6, Unit 9: Sterling volatility and VaR

**Prof. Dr. Mehmet Songur** · ML Lab, Module 6

Data: the sterling/dollar rate, daily since 1971 (FRED: DEXUSUK, the US Federal Reserve's H.10 series, public domain). The setup cell downloads a series called `fx` (date-indexed). **Have Gemini write the code**, then **audit it**.

**Rules**
1. Do not change the **Setup** and **Check** cells; just run them.
2. Generate code with Gemini in the task cells; read it before running.
3. If a check cell passes, it gives you a **completion code**; paste it into the box in ML Lab.

In [ ]:
#@title Setup — run it, do not change it

import base64, warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score
from sklearn.metrics import r2_score
warnings.filterwarnings("ignore")

def _k(s):
    return base64.b64decode(s).decode()

def _referans(seri):
    r = 100 * np.log(seri).diff().dropna()
    t = pd.DataFrame(index=r.index)
    for w in [5, 20, 60]:
        t[f"v{w}"] = np.sqrt((r ** 2).rolling(w).mean())
    t["y_v"] = np.sqrt((r ** 2).rolling(20).mean().shift(-20))
    t = t.dropna()
    X, y = t[["v5", "v20", "v60"]], t["y_v"]
    e = t.index < "2015-01-01"
    out = dict(r=r, t=t,
               cv=cross_val_score(LinearRegression(), X, y, cv=TimeSeriesSplit(5, gap=20), scoring="r2").mean(),
               cv_nogap=cross_val_score(LinearRegression(), X, y, cv=TimeSeriesSplit(5), scoring="r2").mean(),
               cv_rand=cross_val_score(LinearRegression(), X, y, cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean(),
               test=r2_score(y[~e], LinearRegression().fit(X[e], y[e]).predict(X[~e])),
               naive=r2_score(y[~e], t.loc[~e, "v60"]))
    yr = r.shift(-1).reindex(t.index)
    oran = (yr / t["v60"]).dropna()
    ee = oran.index < "2015-01-01"
    q = np.quantile(oran[ee], 0.01)
    out.update(q=q, q_all=np.quantile(oran, 0.01), exc=(yr[~e] < q * t.loc[~e, "v60"]).mean(), exc_n=(yr[~e] < -2.326 * t.loc[~e, "v60"]).mean())
    return out

fx = pd.read_csv("https://raw.githubusercontent.com/mehmetsongur/ml-lab/main/data/fred/DEXUSUK.csv", index_col=0, parse_dates=True).iloc[:, 0]
fx = pd.to_numeric(fx, errors="coerce").dropna()                    # dollars per pound, daily
print(f"fx: {fx.index[0].date()} – {fx.index[-1].date()}, {len(fx)} days")
_R = _referans(fx)

def check_A(r, table, cv_r2, test_r2, naive_r2):
    if len(r) != len(_R["r"]) or not np.allclose(np.asarray(r), _R["r"].values):
        print("❌ r differs from the expected one: r = 100 * np.log(fx).diff().dropna() (percent log return)."); return
    for c in ["v5", "v20", "v60", "y_v"]:
        if c not in table.columns:
            print(f"❌ table has no {c} column."); return
    if len(table) != len(_R["t"]) or not table.index.equals(_R["t"].index):
        print("❌ table's rows differ from the expected ones: apply dropna() after building all the columns."); return
    for c in ["v5", "v20", "v60"]:
        if not np.allclose(table[c].values, _R["t"][c].values):
            print(f"❌ {c} differs from the expected one: np.sqrt((r ** 2).rolling(w).mean()), a backward-looking window only (no center=True)."); return
    if not np.allclose(table["y_v"].values, _R["t"]["y_v"].values):
        print("❌ y_v differs from the expected one: the volatility of the next 20 days, np.sqrt((r ** 2).rolling(20).mean().shift(-20))."); return
    if abs(float(cv_r2) - _R["cv"]) > 1e-6:
        if abs(float(cv_r2) - _R["cv_rand"]) < 1e-3:
            print("❌ cv_r2 was computed with a random split (KFold, shuffle): with time series use TimeSeriesSplit(5, gap=20)."); return
        if abs(float(cv_r2) - _R["cv_nogap"]) < 1e-6 and abs(_R["cv_nogap"] - _R["cv"]) > 1e-6:
            print("❌ cv_r2 has no gap: TimeSeriesSplit(n_splits=5, gap=20)."); return
        print("❌ cv_r2 differs from the expected value: LinearRegression, X = v5, v20, v60; cv=TimeSeriesSplit(n_splits=5, gap=20), scoring='r2', mean."); return
    if abs(float(test_r2) - _R["test"]) > 1e-6:
        print("❌ test_r2 differs from the expected value: the model should be fitted before 2015 only and R² computed from 2015 on."); return
    if abs(float(naive_r2) - _R["naive"]) > 1e-6:
        print("❌ naive_r2 differs from the expected value: r2_score(y, table['v60']) in the test period."); return
    print(f"✅ All checks passed (cv R² {_R['cv']:.3f}, test R² {_R['test']:.3f}, naive v60 {_R['naive']:.3f}). Your completion code: " + _k("TUw2LVZPTC00VFpD"))

def check_B(table, q, exc_rate, exc_normal):
    if "y_r" not in table.columns or not np.allclose(table["y_r"].values, _R["r"].shift(-1).reindex(table.index).values, equal_nan=True):
        print("❌ table['y_r'] should be tomorrow's return: r.shift(-1)."); return
    if abs(float(q) - _R["q"]) > 1e-9:
        if abs(float(q) - _R["q_all"]) < 1e-9:
            print("❌ The multiplier was learned from all the data (test included): use only the 1% quantile of y_r / v60 before 2015."); return
        print("❌ q differs from the expected value: the 1% quantile of (y_r / v60) before 2015, np.quantile(..., 0.01)."); return
    if abs(float(exc_rate) - _R["exc"]) > 1e-9:
        print("❌ exc_rate differs from the expected value: the mean of (y_r < q * v60) from 2015 on."); return
    if abs(float(exc_normal) - _R["exc_n"]) > 1e-9:
        print("❌ exc_normal differs from the expected value: the mean of (y_r < -2.326 * v60) from 2015 on."); return
    print(f"✅ All checks passed (multiplier {_R['q']:.2f}; exceedance rate {100 * _R['exc']:.2f}%, with the normal multiplier {100 * _R['exc_n']:.2f}%). Your completion code: " + _k("TUw2LVZBUi03QkhO"))

fx.tail(3)

## Task A — Volatility forecast

1. Daily log return: `r = 100 * np.log(fx).diff().dropna()`.
2. A DataFrame called `table`: for w = 5, 20, 60, `v{w} = np.sqrt((r ** 2).rolling(w).mean())` (past only); target `y_v = np.sqrt((r ** 2).rolling(20).mean().shift(-20))` (the next 20 days); then `dropna()`.
3. `X = table[["v5", "v20", "v60"]]`, `y = table["y_v"]`. The mean of `cross_val_score(..., cv=TimeSeriesSplit(n_splits=5, gap=20), scoring="r2")` for `LinearRegression`: `cv_r2`.
4. `trn = table.index < "2015-01-01"`, `tst = ~trn`. Fit the model on `trn` only; R² on `tst`: `test_r2`. The R² of `table.loc[tst, "v60"]` as the naive forecast: `naive_r2`.

**Audit questions:** Is there a random split (`train_test_split`, `KFold(shuffle=True)`)? Do the windows look backward? Does the target look forward?

In [ ]:
# Task A: generate the code with Gemini here, read it, then run it.

### Check A — run it, do not change it

In [ ]:
check_A(r, table, cv_r2, test_r2, naive_r2)

## Task B — Value at risk (1% VaR)

1. Add tomorrow's return to the table: `table["y_r"] = r.shift(-1)`.
2. Multiplier: the 1% quantile of the ratio `table["y_r"] / table["v60"]` before 2015: `q = np.quantile(..., 0.01)` (drop missing values).
3. Exceedance: `table["y_r"] < q * table["v60"]`. Its mean from 2015 on: `exc_rate`.
4. Do the same with the normal multiplier (−2.326): `exc_normal`. Compare both with the 1% target and look at how exceedances are spread across years.

**Audit questions:** Was the multiplier learned from the training period only? In which years do exceedances pile up?

In [ ]:
# Task B: generate the code with Gemini here, read it, then run it.

### Check B — run it, do not change it

In [ ]:
check_B(table, q, exc_rate, exc_normal)

## Your audit notes

- Did Gemini work in a time-respecting way? What did you need to fix?
- How do the sterling results compare with euro/dollar? (cv_r2, test_r2, naive_r2)
- How would you explain the VaR results to a risk committee? (exceedance rate, gap with the normal multiplier, limitations)